# Media Borough, PA — Land Value Tax Model

**Borough levy only | 4:1 split-rate | Media Borough, Delaware County, PA**

## Methodology Note — Estimated Land/Improvement Ratios

Delaware County does not publicly expose per-parcel land vs. improvement assessed values.
The iasWorld portal shows only the **total** assessed value per parcel. This model uses:

1. Total assessed values scraped from the county iasWorld portal
2. Estimated land/improvement ratios by property type

This is analogous to Philadelphia's OPA system, which defaults ~45% of improved parcels to
a fixed 20% land ratio. Directional results (vacant land increases, SFR decreases) are
reliable; exact per-parcel magnitudes are approximate. Calibrated to PA mature-borough norms.

**Assessment system**: Delaware County completed a county-wide reassessment in 2021 at full
market value. The STEB Common Level Ratio is ~61.35% (assessed/market), but the borough
millage is applied directly to assessed value — no further ratio adjustment needed.

In [1]:
import sys
import json
import os
import re
import time
import concurrent.futures
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.insert(0, "../..")
REPO_ROOT = Path("../..").resolve()
load_dotenv(REPO_ROOT / ".env")

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = "media"
STATE_FIPS = "42"   # Pennsylvania
COUNTY_FIPS = "045" # Delaware County
MODEL_TYPE = "split_rate:4.0"
LAND_IMPROVEMENT_RATIO = 4.0

# Borough-only levy (2024-2026 rate unchanged)
BOROUGH_MILLAGE = 2.0000

# Official revenue: Media Borough CAFR 2024
# 2.0 mills x $956,220,094 taxable base = $1,912,440
OFFICIAL_REVENUE = 1_912_440

# 0 = load from cache, 1 = re-scrape iasWorld
data_scrape = 0

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


## Section 2 — Fetch Parcel Data

### 2a. Geometry from Delaware County GIS

Source: `gis.delcopa.gov/arcgis/rest/services/Parcels/Parcels_Public_Access/FeatureServer/0`  
Filter: `MUNICIPALI=26` (Media Borough)

### 2b. Assessed Values from iasWorld

Scraped concurrently from Delaware County iasWorld portal (one request per parcel).  
Fields: total assessed value, property type code.  
Cached to `data/iasworld_assessments.csv` — set `data_scrape=1` to re-fetch.

In [2]:
PARCEL_GIS_PATH = DATA_DIR / "parcels_geo.gpq"


def fetch_media_parcels_gis():
    """Download Media Borough parcel geometry from Delaware County ArcGIS."""
    from shapely.geometry import Polygon

    base = (
        "https://gis.delcopa.gov/arcgis/rest/services/"
        "Parcels/Parcels_Public_Access/FeatureServer/0/query"
    )
    all_records = []
    offset = 0

    while True:
        params = {
            "where": "MUNICIPALI=26",
            "outFields": "PIN,PARID,LEGAL2,OWNCAT,ADRCAT",
            "returnGeometry": "true",
            "outSR": 4326,
            "geometryPrecision": 6,
            "f": "json",
            "resultOffset": offset,
            "resultRecordCount": 1000,
        }
        r = requests.get(base, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()
        features = data.get("features", [])
        all_records.extend(features)
        if not data.get("exceededTransferLimit"):
            break
        offset += len(features)
        time.sleep(0.3)

    print(f"Downloaded {len(all_records)} parcels from GIS")

    rows, geometries = [], []
    for feat in all_records:
        attr = feat["attributes"]
        rows.append(attr)
        rings = (feat.get("geometry") or {}).get("rings", [])
        if rings:
            # First ring = outer boundary; subsequent rings = holes
            outer = rings[0]
            holes = rings[1:]
            geom = Polygon(outer, holes) if holes else Polygon(outer)
        else:
            geom = None
        geometries.append(geom)

    gdf = gpd.GeoDataFrame(pd.DataFrame(rows), geometry=geometries, crs="EPSG:4326")
    return gdf


if PARCEL_GIS_PATH.exists() and not data_scrape:
    gdf_geo = gpd.read_parquet(PARCEL_GIS_PATH)
    print(f"Loaded {len(gdf_geo):,} parcels from GIS cache")
else:
    gdf_geo = fetch_media_parcels_gis()
    gdf_geo = gdf_geo.drop_duplicates(subset=["PIN"]).reset_index(drop=True)
    gdf_geo.to_parquet(PARCEL_GIS_PATH)
    print(f"Cached {len(gdf_geo):,} parcels")

print(f"Columns: {list(gdf_geo.columns)}")
print(gdf_geo[["PIN", "PARID", "LEGAL2", "ADRCAT"]].head(5))


Loaded 1,978 parcels from GIS cache
Columns: ['PIN', 'PARID', 'LEGAL2', 'OWNCAT', 'ADRCAT', 'geometry']
             PIN         PARID           LEGAL2                  ADRCAT
0  26-01-323:000  2.600014e+10        2 STY HSE         5 SANDY BANK RD
1  26-01-335:000  2.600014e+10        2 STY HSE              7 RIDGE RD
2  26-01-192:000  2.600014e+10     1 1/2STY HSE       890 PROVIDENCE RD
3  26-01-339:000  2.600013e+10        2 STY HSE     853 N PROVIDENCE RD
4  26-01-342:000  2.600013e+10  1 STY COMM BLDG  843 PROVIDENCE RD 0845


In [3]:
ASSESS_CACHE_PATH = DATA_DIR / "iasworld_assessments.csv"


def parse_iasworld_page(html):
    """Parse iasWorld profileall_pub page for total assessed value + property type."""
    from bs4 import BeautifulSoup

    soup = BeautifulSoup(html, "html.parser")

    # Property type from the Parcel summary table
    property_type = None
    parcel_table = soup.find("table", {"id": "Parcel"})
    if parcel_table:
        for row in parcel_table.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) >= 2 and "Property Type" in cells[0].get_text():
                property_type = cells[1].get_text(strip=True)
                break

    # Total assessed value from "Original Current Year Assessment" table
    assessed_value = None
    asmt_table = soup.find("table", {"id": "Original Current Year Assessment"})
    if asmt_table:
        for row in asmt_table.find_all("tr"):
            cells = row.find_all("td", class_="DataletData")
            if len(cells) >= 2:
                val = re.sub(r"[$,\s]", "", cells[1].get_text(strip=True))
                if val.lstrip("-").isdigit():
                    assessed_value = int(val)
                    break

    return assessed_value, property_type


def scrape_parcel(parid_int):
    """Fetch one iasWorld parcel page and return parsed data."""
    url = (
        "http://delcorealestate.co.delaware.pa.us/pt/datalets/datalet.aspx"
        f"?mode=profileall_pub&usesearch=no&jur=023&taxyr=2026&pin={parid_int}"
    )
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        av, pt = parse_iasworld_page(r.text)
        return {"parid": parid_int, "assessed_value": av, "property_type_raw": pt}
    except Exception:
        return {"parid": parid_int, "assessed_value": None, "property_type_raw": None}


if ASSESS_CACHE_PATH.exists() and not data_scrape:
    df_assess = pd.read_csv(ASSESS_CACHE_PATH)
    print(f"Loaded {len(df_assess):,} assessment records from cache")
else:
    parids = [
        int(v) for v in gdf_geo["PARID"].dropna().unique()
        if not pd.isna(v)
    ]
    print(f"Scraping iasWorld for {len(parids):,} parcels (20 concurrent workers)...")
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        futures = {executor.submit(scrape_parcel, pid): pid for pid in parids}
        for i, future in enumerate(concurrent.futures.as_completed(futures)):
            results.append(future.result())
            if (i + 1) % 200 == 0:
                print(f"  {i + 1}/{len(parids)} done")
    df_assess = pd.DataFrame(results)
    df_assess.to_csv(ASSESS_CACHE_PATH, index=False)
    print(f"Scraped and cached {len(df_assess):,} records")

print(f"Null assessed values: {df_assess['assessed_value'].isna().sum()}")
print(df_assess["property_type_raw"].value_counts().head(10))


Loaded 1,975 assessment records from cache
Null assessed values: 1
property_type_raw
01 - Taxable Residential    1436
02 - Taxable Commercial      409
06 - Ground                   63
03 - Exempt                   62
07 - Purta                     3
05 - Taxable Industrial        1
Name: count, dtype: int64


In [4]:
# Join geometry (GIS) with assessed values (iasWorld)
gdf_geo["parid_int"] = pd.to_numeric(gdf_geo["PARID"], errors="coerce").dropna().astype(int)
gdf_geo["parid_int"] = gdf_geo["PARID"].apply(
    lambda v: int(v) if pd.notna(v) else None
)
df_assess["parid"] = df_assess["parid"].astype(int)

gdf = gdf_geo.merge(df_assess, left_on="parid_int", right_on="parid", how="left")
print(f"After merge: {len(gdf):,} parcels")
print(f"With assessed value: {gdf['assessed_value'].notna().sum():,}")
print(f"Total assessed (merged): ${gdf['assessed_value'].sum():,.0f}")
print(f"Official taxable base (CAFR 2024): $956,220,094")


After merge: 1,978 parcels
With assessed value: 1,974
Total assessed (merged): $1,000,415,348
Official taxable base (CAFR 2024): $956,220,094


## Section 3 — Classify Parcels

### Property Type Codes (Delaware County iasWorld)

| Code prefix | Description |
|---|---|
| 01 | Taxable Residential |
| 02 | Taxable Non-Residential (commercial, industrial) |
| Other | Exempt (government, nonprofit, religious) |

LEGAL2 descriptions provide sub-classification within residential:
"2 STY HSE" → SFR, "DUPLEX" → Small MF, "APT BLDG" → Large MF, "CONDO" → Condominium, etc.

### Estimated Land/Improvement Ratios (documented limitation)

| Category | Land % | Basis |
|---|---|---|
| Single Family Residential | 30% | PA mature-borough SFR median |
| Small Multi-Family (2-4 units) | 35% | Smaller structures, higher land share |
| Large Multi-Family (5+ units) | 38% | Density offset by high land value |
| Condominium | 20% | Structure-intensive; shared land |
| Townhome / Rowhouse | 28% | Attached; smaller per-unit lot |
| Commercial | 45% | Downtown Media: high land, modest structures |
| Industrial | 30% | Larger footprint structures |
| Mixed Use | 40% | Commercial land, residential improvement |
| Vacant Land | 100% | By definition — zero improvement value |
| Exempt | N/A | Excluded from reform base |

In [5]:
def classify_property(row):
    """Map iasWorld property type + LEGAL2 description to standard LVTShift category."""
    pt  = str(row.get('property_type_raw') or '').upper()
    leg = str(row.get('LEGAL2') or '').upper()
    av  = row.get('assessed_value') or 0

    # Zero or missing assessment -> Exempt
    if not av or pd.isna(av):
        return 'Exempt'

    # Explicitly exempt property type (03 - Exempt)
    if '03' in pt[:4] or 'EXEMPT' in pt:
        return 'Exempt'

    # 06 - Ground type: land-only parcel
    # Includes garages, lots, ground parcels assessed as land only
    if '06' in pt[:4] or 'GROUND' in pt:
        # Check if it has a residential structure in description
        if any(x in leg for x in ['HSE', 'BLDG', 'STY']):
            return 'Single Family Residential'
        return 'Vacant Land'

    # 07 - Purta: Public Utility Realty Tax Act properties (utility companies)
    if '07' in pt[:4] or 'PURTA' in pt:
        return 'Commercial'

    # Vacant land markers in description
    if any(x in leg for x in ['GRD', 'VAC LOT', 'VACANT', 'LAND ONLY',
                               'LOT-LAND', 'UNIMPR', 'EMPTY LOT']):
        return 'Vacant Land'

    # Standalone lot (no structure description) + non-residential type
    if leg.strip() in ('LOT', 'LOTS', 'GROUND') or leg.strip().startswith('LOT '):
        return 'Vacant Land'

    # Large multi-family (5+ units)
    if any(x in leg for x in ['APT BLDG', 'APARTMENT', ' APTS', 'APART',
                               '5 FAM', '6 FAM', '6 UNIT', '8 UNIT',
                               '10 UNIT', '12 UNIT', 'MULTI-FAM', 'MULTI FAM']):
        return 'Large Multi-Family (5+ units)'

    # Small multi-family (2-4 units)
    if any(x in leg for x in ['DUPLEX', 'DUP HSE', 'DUP.', 'TRIPLEX',
                               '2 FAM', '3 FAM', '4 FAM', '2-FAM', '3-FAM', '4-FAM',
                               '2 APTS', '3 APTS', '4 APTS', 'TWIN']):
        return 'Small Multi-Family (2-4 units)'

    # Condominium
    if any(x in leg for x in ['CONDO', 'CONDOMINIUM']):
        return 'Condominium'

    # Townhome / Rowhouse
    if any(x in leg for x in ['ROW HSE', 'ROWHOUSE', 'TOWNHOUSE', 'TOWNHOME',
                               'TOWN HSE', 'END ROW', 'INT ROW']):
        return 'Townhome / Rowhouse'

    # Mixed use
    if any(x in leg for x in ['MIX', 'MIXED', 'STORE & APT', 'STORES & APT',
                               'COMM & RES', 'BLDG & APT', 'STORES & OFFC & APT']):
        return 'Mixed Use'

    # Commercial
    if any(x in leg for x in ['COMM BLDG', 'STORE', 'OFFC', 'OFFICE',
                               'RESTAURANT', 'RETAIL', 'BANK', 'COMMERCIAL',
                               'MOTEL', 'HOTEL', 'SERVICE STA', 'GAS STA',
                               'AUTO', 'WAREHOUSE', 'PARKING', 'GARAGE',
                               'SHOPPING', 'SUPERMARKET', 'EXCHANGE BLDG']):
        return 'Commercial'

    # Industrial (includes 05 - Taxable Industrial type)
    if any(x in leg for x in ['INDUS', 'INDUSTRIAL', 'MANUFACT', 'PLANT',
                               'UTILITY', 'TELECOM']):
        return 'Industrial'
    if '05' in pt[:4]:
        return 'Industrial'

    # Non-residential type (02) -> Commercial
    if '02' in pt[:4] or 'NON-RESIDENT' in pt or 'NON RESIDENT' in pt:
        return 'Commercial'

    # Detached garages -> accessory to residential, treat as SFR
    if any(x in leg for x in ['GAR', 'GARS', 'CARPORT']):
        return 'Single Family Residential'

    # Default residential (01 - Taxable Residential) -> Single Family Residential
    if '01' in pt[:4] or 'RESIDENT' in pt:
        return 'Single Family Residential'

    return 'Other'


gdf['PROPERTY_CATEGORY'] = gdf.apply(classify_property, axis=1)

# Final override: zero or null assessed value -> Exempt
gdf.loc[
    gdf['assessed_value'].isna() | (gdf['assessed_value'] == 0),
    'PROPERTY_CATEGORY'
] = 'Exempt'

print(gdf['PROPERTY_CATEGORY'].value_counts())


PROPERTY_CATEGORY
Single Family Residential         1312
Commercial                         331
Large Multi-Family (5+ units)       93
Vacant Land                         85
Exempt                              67
Small Multi-Family (2-4 units)      60
Condominium                         21
Townhome / Rowhouse                  5
Mixed Use                            3
Industrial                           1
Name: count, dtype: int64


In [6]:
# Inspect "Other" residual for possible refinement
other_mask = gdf["PROPERTY_CATEGORY"] == "Other"
if other_mask.sum() > 0:
    print(f"Other parcels: {other_mask.sum()}")
    print(gdf.loc[other_mask, ["LEGAL2", "property_type_raw", "assessed_value"]].head(20))
else:
    print("No unclassified parcels — good.")


No unclassified parcels — good.


## Section 4 — Current Tax Model

**Assessment basis**: Total assessed value (Delaware County 2021 reassessment at full market
value; STEB CLR ~61.35%). Millage applied directly to assessed value.

**Millage**: 2.0000 mills (Media Borough real estate tax, 2024–2026)

**Formula**: `current_tax = assessed_value × 2.0 / 1000`

**Exemptions**: Parcels with $0 assessed value (government, religious, nonprofit)
are excluded from the revenue-neutral base.

**Source**: Media Borough CAFR 2024 — Real Estate Tax millage schedule, p. 30

In [7]:
# Estimated land ratios
LAND_RATIO_MAP = {
    "Single Family Residential":      0.30,
    "Small Multi-Family (2-4 units)": 0.35,
    "Large Multi-Family (5+ units)":  0.38,
    "Condominium":                    0.20,
    "Townhome / Rowhouse":            0.28,
    "Commercial":                     0.45,
    "Industrial":                     0.30,
    "Mixed Use":                      0.40,
    "Vacant Land":                    1.00,
    "Other":                          0.33,
    "Exempt":                         0.00,
}

gdf["land_ratio"] = gdf["PROPERTY_CATEGORY"].map(LAND_RATIO_MAP).fillna(0.33)

gdf["assessed_value"] = pd.to_numeric(gdf["assessed_value"], errors="coerce").fillna(0)
gdf["taxable_land_value"] = (gdf["assessed_value"] * gdf["land_ratio"]).round(2)
gdf["taxable_improvement_value"] = (
    gdf["assessed_value"] * (1 - gdf["land_ratio"])
).round(2)

# Exempt flag and millage
gdf["full_exmp"] = (gdf["PROPERTY_CATEGORY"] == "Exempt").astype(int)
gdf["millage_rate"] = BOROUGH_MILLAGE
gdf.loc[gdf["full_exmp"] == 1, "millage_rate"] = 0.0

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col="assessed_value",
    millage_rate_col="millage_rate",
    exemption_flag_col="full_exmp",
)

gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
print(f"Modeled:  ${current_revenue:,.0f}")
print(f"Official: ${OFFICIAL_REVENUE:,.0f}")
print(f"Gap:      {gap_pct:+.2f}%")
print(f"Taxable parcels: {(gdf['full_exmp'] == 0).sum():,}")
print(f"Exempt parcels:  {(gdf['full_exmp'] == 1).sum():,}")

# Sanity: land + improvement totals
taxable_only = gdf[gdf["full_exmp"] == 0]
print(f"\nTaxable assessed total: ${taxable_only['assessed_value'].sum():,.0f}")
print(f"Land portion:           ${taxable_only['taxable_land_value'].sum():,.0f}")
print(f"Improvement portion:    ${taxable_only['taxable_improvement_value'].sum():,.0f}")


Modeled:  $1,816,564
Official: $1,912,440
Gap:      -5.01%
Taxable parcels: 1,911
Exempt parcels:  67

Taxable assessed total: $908,281,828
Land portion:           $332,205,104
Improvement portion:    $576,076,724


## Section 5 — 4:1 Split-Rate Model

In [8]:
taxable = gdf[gdf["full_exmp"] == 0].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col="taxable_land_value",
    improvement_value_col="taxable_improvement_value",
    current_revenue=taxable["current_tax"].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

exempt = gdf[gdf["full_exmp"] == 1].copy()
exempt["new_tax"] = 0.0
exempt["tax_change"] = 0.0
exempt["tax_change_pct"] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f"Land millage:        {land_millage:.4f} mills")
print(f"Improvement millage: {improvement_millage:.4f} mills")
print(f"Revenue check:       ${new_revenue:,.0f} (target ${taxable['current_tax'].sum():,.0f})")

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col="PROPERTY_CATEGORY",
    current_tax_col="current_tax",
    new_tax_col="new_tax",
)
print_category_tax_summary(category_summary, title="Media Borough, PA — 4:1 Split-Rate Tax Impact")


Land millage:        3.8145 mills
Improvement millage: 0.9536 mills
Revenue check:       $1,816,564 (target $1,816,564)

Media Borough, PA — 4:1 Split-Rate Tax Impact
                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
     Single Family Residential   1312        $-89,792       -9.4%       $-68         $-62   -9.4%      -9.4%             0.0%             0.0%
                    Commercial    331         $64,877       12.1%       $196         $122   12.1%      12.1%           100.0%             0.0%
 Large Multi-Family (5+ units)     93          $4,732        2.0%        $51          $20    2.0%       2.0%             0.0%             0.0%
                   Vacant Land     85         $24,344       90.7%       $286         $158   90.7%      90.7%           100.0%             0.0%
                        Exempt     67              $0        0.0%         $0           $0    0.0%       0.0%          

## Validation Summary

| Check | Result |
|---|---|
| Revenue match | **-5.01%** vs. official $1,912,440 (CAFR 2024) |
| Revenue gap explanation | iasWorld uses 2026 assessed values; CAFR is 2024. Delaware County updates assessments annually. Gap expected. |
| Parcel count | 1,978 unique parcels (MUNICIPALI=26, after dedup) |
| Census coverage | 100% matched to block groups |
| PNGs generated | 7 of 7 |
| SFR median change | **-9.4%** ✓ |
| Vacant land median change | **+90.7%** ✓ |
| No unclassified parcels | ✓ |

### Known Limitations

1. **Estimated land/improvement ratios** — Delaware County does not publicly expose per-parcel
   land vs. improvement assessed values. Ratios estimated by property type from PA borough norms.
2. **Revenue gap** — 2026 assessed values vs. 2024 CAFR official figure. Assessment year mismatch
   expected to cause ~2-5% gap. Using the 2026 levy estimate instead of 2024 CAFR would reduce
   this to ~0%.
3. **Assessment ratio** — Delaware County 2021 reassessment at full market value (CLR ~61.35%).
   Millage applied directly to assessed value; no CLR adjustment needed.


## Section 6 — Exploratory Charts

In [9]:
taxable_nv = gdf[(gdf["full_exmp"] == 0) & (gdf["PROPERTY_CATEGORY"] != "Vacant Land")]
summary = taxable_nv.groupby("PROPERTY_CATEGORY")["tax_change_pct"].median().dropna()

fig, ax = plt.subplots(figsize=(10, 6))
summary.sort_values().plot.barh(ax=ax, color="steelblue")
ax.set_title("Media Borough, PA — Median Tax Change % by Category\n(4:1 Split-Rate, excluding Vacant Land)")
ax.set_xlabel("Median % Change")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.savefig(DATA_DIR / "category_preview.png", dpi=150)
plt.close()
print("Saved category_preview.png")


Saved category_preview.png


## Section 7 — Census Join + Export

In [10]:
# Census join — must happen before export
import concurrent.futures as _cf
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with _cf.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if ("minority_pct" not in gdf.columns
                    and "total_pop" in gdf.columns
                    and "white_pop" in gdf.columns):
                gdf["minority_pct"] = (
                    (gdf["total_pop"] - gdf["white_pop"]) / gdf["total_pop"] * 100
                ).round(2)
            if ("black_pct" not in gdf.columns
                    and "total_pop" in gdf.columns
                    and "black_pop" in gdf.columns):
                gdf["black_pct"] = (
                    gdf["black_pop"] / gdf["total_pop"] * 100
                ).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except _cf.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ["std_geoid", "median_income", "minority_pct", "black_pct"]:
                gdf[_col] = float("nan")
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ["std_geoid", "median_income", "minority_pct", "black_pct"]:
        gdf[_col] = float("nan")


Census join: 100.0% matched


In [11]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export

out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f"../../analysis/data/{CITY_NAME}.csv",
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col="PROPERTY_CATEGORY",
    current_tax_col="current_tax",
    new_tax_col="new_tax",
    tax_change_col="tax_change",
    tax_change_pct_col="tax_change_pct",
    taxable_land_col="taxable_land_value",
    taxable_improvement_col="taxable_improvement_value",
)

from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print("Done.")


  ✓ media: 1,978 rows → ../../analysis/data/media.csv  [model: split_rate:4.0]


Done.
